# Video Anomaly Detection — Tesla T4 Colab Training

Before running this notebook, select **Runtime → Change runtime type → T4 GPU**.

Place this project in Google Drive. The expected dataset path is:

`Course Project/data/raw/ucf_crime/Train/<class>/...`

`Course Project/data/raw/ucf_crime/Test/<class>/...`

The folders may contain PNG frames, as in your current dataset.

In [ ]:
# Mount Google Drive and enter the project folder.
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Course Project
!pwd
!ls

In [ ]:
# Confirm that Colab assigned a CUDA GPU. It should report Tesla T4.
!nvidia-smi
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU found. Change the Colab runtime to T4 GPU and reconnect.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Install non-PyTorch dependencies and test R3D-18 on the GPU.
# This intentionally preserves Colab's CUDA-enabled PyTorch installation.
!bash scripts/setup_colab.sh

In [ ]:
# Check the expected dataset folders before creating the training lists.
from pathlib import Path
dataset_root = Path('data/raw/ucf_crime')
for split in ('Train', 'Test'):
    split_path = dataset_root / split
    assert split_path.is_dir(), f'Missing: {split_path.resolve()}'
    classes = sorted(path.name for path in split_path.iterdir() if path.is_dir())
    print(f'{split}: {len(classes)} classes — {classes}')

In [ ]:
# Generate data/subset/ucf_crime_train_subset.txt and ucf_crime_test_subset.txt.
!python scripts/setup_environment.py

In [ ]:
# Start training. A tqdm progress bar appears below this cell.
# If CUDA runs out of memory, change --batch-size 8 to --batch-size 4.
!python scripts/train_supervised.py --data-root data/raw/ucf_crime --train-list data/subset/ucf_crime_train_subset.txt --val-list data/subset/ucf_crime_test_subset.txt --epochs 30 --patience 5 --batch-size 8 --num-workers 2

In [ ]:
# Display the saved AUC-ROC curve and the final metrics.
from IPython.display import Image, display
from pathlib import Path
import json

roc_file = Path('results/roc_curve.png')
metrics_file = Path('results/metrics.json')
if roc_file.exists():
    display(Image(filename=str(roc_file)))
else:
    print('ROC curve has not been created yet.')
if metrics_file.exists():
    print(json.dumps(json.loads(metrics_file.read_text()), indent=2))
else:
    print('Metrics have not been created yet.')